# Mamdani in Context: Money, Media, and the Coalitions of 2017 and 2021

Part 10 established that Mamdani's November 2025 vote held the outer-borough neighborhoods that had swung to Trump and lost the affluent ones. This notebook puts that result against the two prior mayoral cycles on three dimensions:

1. **Money.** Candidate-committee spending and independent expenditures for and against each major candidate, 2013–2025, from the NYC Campaign Finance Board.
2. **Media.** Daily coverage volume and tone for each major candidate by news source, from the GDELT DOC API, for the 2021 and 2025 cycles (the API begins in 2017; the 2017 race was uncompetitive).
3. **Coalitions.** The same election districts across the 2017 and 2021 generals and Democratic primaries (crosswalked from the 2012 district lines through census blocks), so Mamdani's map can be compared with de Blasio's and Adams's.

**Inputs:** `data/raw/nyc_cfb/*.csv`, `data/raw/gdelt_nyc/gdelt_nyc_daily.csv`, `data/raw/nyc/nyc_ed_history.csv`, `data/raw/nyc/nyc_ed_panel.csv`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables'); FIGURES_DIR = Path('../outputs/figures')
pd.set_option('display.width', 220)

## 1. Money: candidate committees and independent expenditures, 2013–2025

In [2]:
ie = pd.read_csv(RAW_DIR / 'nyc_cfb' / 'independent_expenditures_by_candidate.csv')
fin = pd.read_csv(RAW_DIR / 'nyc_cfb' / 'mayor_candidate_finance_summary.csv')
m = ie[ie.office == 'Mayor'].groupby(['cycle', 'candidate'])[['support', 'oppose']].sum()
m['net_ie'] = m.support - m.oppose
m = m[(m.support + m.oppose) > 250000].sort_values(['cycle', 'support'], ascending=[True, False])
print('Independent expenditures for and against mayoral candidates ($M):'); print((m / 1e6).round(2))
m.to_csv(TABLES_DIR / 'nyc_ie_by_candidate.csv')
fin['cand'] = fin.cand_name.str.replace(r' \(ID.*', '', regex=True)
print('\nCandidate committees ($M): contributions, public funds, spending; contributors; small-dollar share of contributions')
print(fin[['cycle', 'cand', 'net_cntns', 'pubfnd_pmt', 'net_expnd', 'cntrs_no', 'small_share']].to_string(index=False))
# the winner's side vs the best-funded opponent side, each cycle: candidate spending + IE support + IE against the other
def side(cycle, cand_fin, cand_ie):
    f = fin[(fin.cycle == cycle) & fin.cand.str.contains(cand_fin)].net_expnd.sum()
    s = m.loc[(cycle, cand_ie), 'support'] / 1e6 if (cycle, cand_ie) in m.index else 0
    o = m.loc[(cycle, cand_ie), 'oppose'] / 1e6 if (cycle, cand_ie) in m.index else 0
    return f, s, o
rows = []
for cycle, a, ai, b, bi in [(2013, 'de Blasio', 'de Blasio, Bill', 'Quinn', 'Quinn, Christine C'), (2013, 'de Blasio', 'de Blasio, Bill', 'Thompson', 'Thompson, Jr., William C'),
                             (2021, 'Adams, Eric', 'Adams, Eric L', 'Garcia', 'Garcia, Kathryn A'), (2021, 'Adams, Eric', 'Adams, Eric L', 'Yang', 'Yang, Andrew'),
                             (2025, 'Mamdani', 'Mamdani, Zohran K', 'Cuomo', 'Cuomo, Andrew M')]:
    fa, sa, oa = side(cycle, a, ai); fb, sb, ob = side(cycle, b, bi)
    rows.append({'cycle': cycle, 'winner': a, 'winner_committee': fa, 'ie_for_winner': sa, 'ie_against_winner': oa, 'opponent': b, 'opponent_committee': fb, 'ie_for_opponent': sb, 'ie_against_opponent': ob,
                 'winner_side_total': fa + sa + ob, 'opponent_side_total': fb + sb + oa})
sides = pd.DataFrame(rows); sides['opponent_to_winner_ratio'] = sides.opponent_side_total / sides.winner_side_total
print('\nWinner side vs opponent side, $M (committee spending + IE for + IE against the other):'); print(sides.round(2).to_string(index=False))
sides.to_csv(TABLES_DIR / 'nyc_money_sides.csv', index=False)
top = ie[(ie.cycle == 2025) & (ie.office == 'Mayor')].groupby('spender')[['support', 'oppose']].sum()
top['total'] = top.support + top.oppose
print('\n2025 top independent spenders on the mayoral race ($M):'); print((top.sort_values('total', ascending=False).head(12) / 1e6).round(2))

Independent expenditures for and against mayoral candidates ($M):
                                support  oppose  net_ie
cycle candidate                                        
2013  Thompson, Jr., William C     3.08    0.00    3.08
      Quinn, Christine C           0.83    1.00   -0.17
      Lhota, Joseph J              0.52    1.04   -0.52
      de Blasio, Bill              0.51    0.02    0.50
      Liu, John C                  0.45    0.00    0.45
2021  Adams, Eric L                8.04    0.01    8.02
      Donovan, Shaun               6.49    0.00    6.49
      McGuire, Raymond J           5.75    0.00    5.75
      Stringer, Scott M            4.75    0.00    4.75
      Yang, Andrew                 3.85    0.17    3.68
      Wiley, Maya D                1.98    0.00    1.98
2025  Cuomo, Andrew M             31.02    1.67   29.35
      Mamdani, Zohran K            4.83   21.93  -17.10
      Sliwa, Curtis A              0.79    2.82   -2.03
      Adams, Adrienne E            0.5

## 2. Media: coverage volume and tone by source, 2021 vs 2025

GDELT's DOC API returns, per day, the share of all monitored articles matching a query ("volume intensity") and the average tone of those articles (positive minus negative word share, roughly −10 to +10; news is usually negative, so the comparison is relative). Queries are the candidate's name restricted to a source domain; `ALL` is every English-language source GDELT monitors. Tone is averaged over the cycle with daily volume as the weight. Domains GDELT does not index (the pull records them in `empty_series.txt` / `domain_probe.json`) are absent from the tables.

Two coverage gaps in GDELT itself, found in the pull: the index has **no articles from `nypost.com` in 2021** (every 2021 Post query is empty; 2025 is fine), and **no articles at all for June 15 – July 1, 2025** (every 2025 series has 292 of 309 days, the same 17 missing), which is the primary week. The "primary window" below is therefore effectively April 1 – June 14 and the "general window" July 2 – November 4. `nytimes.com` is present in both years but its 2025 volume is a tenth of 2021's (12–18 covered days), and `msnbc.com` is thin in both; treat those columns as unmeasured.


In [3]:
GDELT_OK = (RAW_DIR / 'gdelt_nyc' / 'gdelt_nyc_daily.csv').exists()
if not GDELT_OK:
    print('GDELT file not yet on disk; media section skipped'); raise SystemExit
g = pd.read_csv(RAW_DIR / 'gdelt_nyc' / 'gdelt_nyc_daily.csv')
g['date'] = pd.to_datetime(g.date.astype(str).str[:8])
g = g[g.volume.notna()]
DOMS = [d for d in ['nypost.com', 'nydailynews.com', 'foxnews.com', 'cnn.com', 'msnbc.com', 'nytimes.com', 'ALL'] if d in set(g.domain)]
have = g.groupby(['cycle', 'candidate']).domain.nunique()
print('Series on disk (candidate: domains):'); print(have.to_string())
for f in ['empty_series.txt', 'domain_probe.json']:
    if (RAW_DIR / 'gdelt_nyc' / f).exists(): print(f'\n{f}:', (RAW_DIR / 'gdelt_nyc' / f).read_text().strip())
def wtone(s):
    w = s.volume.fillna(0); t = s.tone
    ok = t.notna() & (w > 0)
    return np.average(t[ok], weights=w[ok]) if ok.any() else np.nan
def summarize(s):
    return pd.Series({'mean_volume': s.volume.mean(), 'days_covered': int((s.volume > 0).sum()), 'tone': wtone(s)})
summ = g.groupby(['cycle', 'candidate', 'domain']).apply(summarize)
tone_tab = summ['tone'].unstack('domain')[DOMS]; vol_tab = summ['mean_volume'].unstack('domain')[DOMS]
print('\nVolume-weighted mean tone by candidate and source, full cycle (Jan 1 to election eve):'); print(tone_tab.round(2))
print('\nMean daily volume intensity (share of all monitored articles, x1000):'); print((vol_tab * 1000).round(3))
tone_tab.to_csv(TABLES_DIR / 'gdelt_nyc_tone.csv'); vol_tab.to_csv(TABLES_DIR / 'gdelt_nyc_volume.csv')
# windows: primaries and generals in each cycle
WIN = {(2021, 'primary'): ('2021-04-01', '2021-06-22'), (2021, 'general'): ('2021-06-23', '2021-11-02'),
       (2025, 'primary'): ('2025-04-01', '2025-06-24'), (2025, 'general'): ('2025-06-25', '2025-11-04')}
wt = {}
for (cyc, lab), (a, b) in WIN.items():
    s = g[(g.cycle == cyc) & (g.date >= a) & (g.date <= b)]
    t = s.groupby(['candidate', 'domain']).apply(wtone).unstack('domain').reindex(columns=DOMS)
    v = s.groupby(['candidate', 'domain']).volume.mean().unstack('domain').reindex(columns=DOMS)
    wt[(cyc, lab)] = t
    print(f'\n{cyc} {lab} window ({a} to {b}): volume-weighted tone'); print(t.round(2))
    print(f'{cyc} {lab} window: mean daily volume x1000'); print((v * 1000).round(3))
    t.to_csv(TABLES_DIR / f'gdelt_nyc_tone_{cyc}_{lab}.csv'); v.to_csv(TABLES_DIR / f'gdelt_nyc_volume_{cyc}_{lab}.csv')


Series on disk (candidate: domains):
cycle  candidate
2021   adams        6
       garcia       6
       sliwa        5
       stringer     6
       wiley        6
       yang         6
2025   adams        7
       cuomo        7
       lander       7
       mamdani      7
       sliwa        7
       stringer     7

empty_series.txt: 2021,adams,nypost.com
2021,garcia,nypost.com
2021,wiley,nypost.com
2021,yang,nypost.com
2021,sliwa,msnbc.com
2021,sliwa,nypost.com
2021,stringer,nypost.com

domain_probe.json: {
 "nypost.com": false,
 "nydailynews.com": false
}

Volume-weighted mean tone by candidate and source, full cycle (Jan 1 to election eve):
domain           nypost.com  nydailynews.com  foxnews.com  cnn.com  msnbc.com  nytimes.com   ALL
cycle candidate                                                                                 
2021  adams             NaN            -1.42        -2.52    -1.46      -1.39        -1.34 -1.86
      garcia            NaN            -1.02        -1.7

In [4]:
if not GDELT_OK: raise SystemExit
# Tone gap: winner minus main opponent, by source, in each window; only pairs with both series on disk
def gap(cycle, a, b, lab):
    t = wt[(cycle, lab)]
    if a not in t.index or b not in t.index: return None
    return (t.loc[a] - t.loc[b]).rename(f'{cycle} {lab}: {a} - {b}')
pairs = [(2021, 'adams', 'garcia', 'primary'), (2021, 'adams', 'wiley', 'primary'), (2021, 'adams', 'yang', 'primary'), (2021, 'adams', 'sliwa', 'general'),
         (2025, 'mamdani', 'cuomo', 'primary'), (2025, 'mamdani', 'lander', 'primary'), (2025, 'mamdani', 'cuomo', 'general'), (2025, 'mamdani', 'sliwa', 'general')]
gaps = pd.concat([x for x in (gap(*p) for p in pairs) if x is not None], axis=1)
print('Tone gap, winner minus opponent, by source (negative = winner covered more negatively):'); print(gaps.round(2))
gaps.to_csv(TABLES_DIR / 'gdelt_nyc_tone_gaps.csv')
# Same candidate, same source, across cycles: the Democratic nominee's coverage in 2021 vs 2025, and the volume ratio
if {'adams', 'mamdani'} <= set(tone_tab.index.get_level_values('candidate')):
    x = pd.DataFrame({'Adams 2021 tone': tone_tab.loc[(2021, 'adams')], 'Mamdani 2025 tone': tone_tab.loc[(2025, 'mamdani')],
                      'Adams 2021 volume x1000': vol_tab.loc[(2021, 'adams')] * 1000, 'Mamdani 2025 volume x1000': vol_tab.loc[(2025, 'mamdani')] * 1000})
    x['volume ratio M/A'] = x['Mamdani 2025 volume x1000'] / x['Adams 2021 volume x1000']
    print('\nDemocratic nominee, 2021 vs 2025, full cycle:'); print(x.round(3))
    x.to_csv(TABLES_DIR / 'gdelt_nyc_nominee_2021_vs_2025.csv')
# Mamdani tone over time, weekly, by source
colors = {'nypost.com': 'tab:red', 'nydailynews.com': 'tab:orange', 'foxnews.com': 'darkred', 'cnn.com': 'tab:blue', 'msnbc.com': 'tab:purple', 'nytimes.com': 'black', 'ALL': 'grey'}
fig, axes = plt.subplots(2, 1, figsize=(11, 7.5), sharex=True)
for dom in DOMS:
    s = g[(g.cycle == 2025) & (g.candidate == 'mamdani') & (g.domain == dom)].set_index('date').sort_index()
    if (s.volume > 0).sum() < 50: continue  # nytimes.com and msnbc.com: a handful of covered days, pure noise weekly
    wk_t = s.resample('W').apply(lambda x: wtone(x) if len(x) else np.nan)
    wk_v = s.volume.resample('W').mean() * 1000
    axes[0].plot(wk_t.index, wk_t.values, label=dom, color=colors[dom], lw=2.5 if dom == 'ALL' else 1.5, alpha=0.9)
    axes[1].plot(wk_v.index, wk_v.values, label=dom, color=colors[dom], lw=2.5 if dom == 'ALL' else 1.5, alpha=0.9)
for ax in axes:
    ax.axvline(pd.Timestamp('2025-06-24'), color='k', ls='--', lw=0.8); ax.grid(alpha=0.3)
    ax.axvspan(pd.Timestamp('2025-06-15'), pd.Timestamp('2025-07-01'), color='0.85', zorder=0)
axes[0].axhline(0, color='k', lw=0.5); axes[0].set_ylabel('GDELT tone (weekly, volume-weighted)'); axes[0].set_title('Coverage of Mamdani by source, 2025 (dashed line: June 24 primary; shaded: no GDELT data June 15 - July 1)'); axes[0].legend(fontsize=8, ncol=4)
axes[1].set_ylabel('volume intensity x1000 (weekly mean)'); axes[1].set_yscale('log')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'gdelt_mamdani_tone_2025.png', dpi=150); print('saved figure')


Tone gap, winner minus opponent, by source (negative = winner covered more negatively):
                 2021 primary: adams - garcia  2021 primary: adams - wiley  2021 primary: adams - yang  2021 general: adams - sliwa  2025 primary: mamdani - cuomo  2025 primary: mamdani - lander  \
domain                                                                                                                                                                                               
nypost.com                                NaN                          NaN                         NaN                          NaN                           1.00                            0.35   
nydailynews.com                         -0.22                        -0.34                        0.19                         0.07                           0.22                           -0.29   
foxnews.com                             -0.39                        -1.00                        0.00                  

## 3. Coalitions: the same districts in 2017, 2021, and 2025

2017 and 2021 results are on the 2012 district lines and are crosswalked to current districts through 2020 census blocks (population-weighted). The comparison is of each winner's share against the demographic type of the district and against the 2020→2024 presidential swing.

In [5]:
d = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_panel.csv', index_col='ElectDist')
h = pd.read_csv(RAW_DIR / 'nyc' / 'nyc_ed_history.csv', index_col='ElectDist')
d = d.join(h, how='left')
d['pres24_2p'] = d.pres24_harris + d.pres24_trump; d['pres20_2p'] = d.pres20_biden + d.pres20_trump; d['gen25_2p'] = d.gen25_mamdani + d.gen25_cuomo
d['trump_share'] = d.pres24_trump / d.pres24_2p; d['swing_to_trump'] = d.trump_share - d.pres20_trump / d.pres20_2p
d['mamdani_gen_share'] = d.gen25_mamdani / d.gen25_total
d['mamdani_h2h_prim'] = d.prim25_h2h_mamdani / (d.prim25_h2h_mamdani + d.prim25_h2h_cuomo)
d['deblasio17_share'] = d.gen17_deblasio / d.gen17_total
d['deblasio17_prim_share'] = d.prim17_deblasio / d.prim17_total
d['adams21_share'] = d.gen21_adams / d.gen21_total
d['adams_h2h_garcia'] = d.prim21_h2h_adams_v_garcia / (d.prim21_h2h_adams_v_garcia + d.prim21_h2h_garcia_v_adams)
d['adams_h2h_wiley'] = d.prim21_h2h_adams_v_wiley / (d.prim21_h2h_adams_v_wiley + d.prim21_h2h_wiley_v_adams)
d['wiley_first_share'] = d.prim21_first_wiley / d.prim21_ballots
d['adams_first_share'] = d.prim21_first_adams / d.prim21_ballots
ok = (d.pres24_2p >= 50) & (d.gen25_2p >= 50) & (d.pres20_2p >= 50) & (d.gen21_total >= 50) & (d.gen17_total >= 50) & (~d.combined_into.fillna(False)) & d.pop2020.gt(0)
e = d[ok].copy(); print('EDs usable across 2017, 2020, 2021, 2024, 2025:', len(e))
e['majority'] = np.where(e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].max(axis=1) >= 0.5,
                         e[['hispanic_share', 'white_nh_share', 'black_nh_share', 'asian_nh_share']].idxmax(axis=1).str.replace('_share', '').str.replace('_nh', ''), 'mixed')
e['swing_decile'] = pd.qcut(e.swing_to_trump, 10, labels=False) + 1
e['income_tercile'] = pd.qcut(e.med_income, 3, labels=['low', 'mid', 'high'])
def agg(s):
    return pd.Series({'N': len(s), 'deBlasio_gen17': s.gen17_deblasio.sum()/s.gen17_total.sum(), 'deBlasio_prim17': s.prim17_deblasio.sum()/s.prim17_total.sum(),
        'Adams_gen21': s.gen21_adams.sum()/s.gen21_total.sum(), 'Adams_v_Garcia_prim21': s.prim21_h2h_adams_v_garcia.sum()/(s.prim21_h2h_adams_v_garcia.sum()+s.prim21_h2h_garcia_v_adams.sum()),
        'Adams_v_Wiley_prim21': s.prim21_h2h_adams_v_wiley.sum()/(s.prim21_h2h_adams_v_wiley.sum()+s.prim21_h2h_wiley_v_adams.sum()), 'Wiley_first21': s.prim21_first_wiley.sum()/s.prim21_ballots.sum(),
        'Harris_24': s.pres24_harris.sum()/s.pres24_2p.sum(), 'Mamdani_gen25': s.gen25_mamdani.sum()/s.gen25_total.sum(), 'Mamdani_v_Cuomo_prim25': s.prim25_h2h_mamdani.sum()/(s.prim25_h2h_mamdani.sum()+s.prim25_h2h_cuomo.sum()),
        'votes_gen17': s.gen17_total.sum(), 'votes_gen21': s.gen21_total.sum(), 'votes_gen25': s.gen25_total.sum(), 'votes_prim21': s.prim21_ballots.sum(), 'votes_prim25': s.prim25_ballots.sum()})
by_maj = e.groupby('majority').apply(agg); print('By majority group:'); print(by_maj.round(3).T)
by_dec = e.groupby('swing_decile').apply(agg); print('\nBy 2020->2024 swing decile:'); print(by_dec.round(3).T)
by_inc = e.groupby('income_tercile').apply(agg); print('\nBy income tercile:'); print(by_inc.round(3).T)
by_maj.to_csv(TABLES_DIR / 'nyc_history_by_majority.csv'); by_dec.to_csv(TABLES_DIR / 'nyc_history_by_swing_decile.csv'); by_inc.to_csv(TABLES_DIR / 'nyc_history_by_income.csv')

EDs usable across 2017, 2020, 2021, 2024, 2025: 3790
By majority group:
majority                     asian       black    hispanic       mixed       white
N                          215.000     624.000     705.000    1029.000    1217.000
deBlasio_gen17               0.542       0.920       0.831       0.685       0.501
deBlasio_prim17              0.643       0.897       0.835       0.739       0.605
Adams_gen21                  0.516       0.909       0.758       0.669       0.578
Adams_v_Garcia_prim21        0.457       0.841       0.624       0.500       0.294
Adams_v_Wiley_prim21         0.566       0.713       0.599       0.507       0.456
Wiley_first21                0.126       0.200       0.204       0.247       0.215
Harris_24                    0.530       0.899       0.702       0.693       0.617
Mamdani_gen25                0.465       0.636       0.584       0.578       0.414
Mamdani_v_Cuomo_prim25       0.614       0.438       0.553       0.646       0.562
votes_gen17    

In [6]:
# Which past coalition does Mamdani's resemble? ED-level correlations of winner shares, weighted by 2025 votes
def wcorr(x, y, w):
    m = x.notna() & y.notna() & w.notna(); x, y, w = x[m], y[m], w[m]
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    return np.average((x-mx)*(y-my), weights=w) / np.sqrt(np.average((x-mx)**2, weights=w) * np.average((y-my)**2, weights=w))
cols = ['deblasio17_share', 'deblasio17_prim_share', 'adams21_share', 'adams_h2h_garcia', 'adams_h2h_wiley', 'wiley_first_share', 'harris_share', 'mamdani_gen_share', 'mamdani_h2h_prim']
e['harris_share'] = e.pres24_harris / e.pres24_2p
C = pd.DataFrame({a: {b: wcorr(e[a], e[b], e.gen25_total) for b in cols} for a in cols})
print('ED-level correlation matrix (weighted by 2025 votes):'); print(C.round(2))
C.to_csv(TABLES_DIR / 'nyc_history_correlations.csv')
# Turnout across cycles by group
print('\nGeneral-election votes by majority group, 2017 / 2021 / 2025 (thousands) and 2025/2021 ratio:')
t = by_maj[['votes_gen17', 'votes_gen21', 'votes_gen25']] / 1000; t['ratio_25_21'] = by_maj.votes_gen25 / by_maj.votes_gen21; print(t.round(2))
print('\nPrimary ballots by majority group, 2021 / 2025 (thousands):'); print((by_maj[['votes_prim21', 'votes_prim25']] / 1000).round(1).assign(ratio=(by_maj.votes_prim25 / by_maj.votes_prim21).round(2)))

ED-level correlation matrix (weighted by 2025 votes):
                       deblasio17_share  deblasio17_prim_share  adams21_share  adams_h2h_garcia  adams_h2h_wiley  wiley_first_share  harris_share  mamdani_gen_share  mamdani_h2h_prim
deblasio17_share                   1.00                   0.94           0.91              0.30            -0.19               0.48          0.79               0.80              0.26
deblasio17_prim_share              0.94                   1.00           0.82              0.43            -0.03               0.34          0.69               0.70              0.16
adams21_share                      0.91                   0.82           1.00              0.13            -0.24               0.42          0.87               0.72              0.19
adams_h2h_garcia                   0.30                   0.43           0.13              1.00             0.81              -0.39         -0.08              -0.09             -0.54
adams_h2h_wiley                

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, (col, lab) in zip(axes, [('deblasio17_share', 'de Blasio, Nov 2017'), ('adams21_share', 'Adams, Nov 2021'), ('mamdani_gen_share', 'Mamdani, Nov 2025')]):
    for grp, c in [('hispanic', 'tab:orange'), ('black', 'tab:purple'), ('white', 'tab:blue'), ('asian', 'tab:green'), ('mixed', 'grey')]:
        s = e[e.majority == grp]
        ax.scatter(s.swing_to_trump, s[col], s=np.sqrt(s.pres24_2p)/2, alpha=0.3, c=c, label=grp, edgecolors='none')
    ax.set_title(lab); ax.set_xlabel('Swing to Trump 2020->2024'); ax.set_xlim(-0.15, 0.45); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.axvline(0, color='k', lw=0.5)
axes[0].set_ylabel('Winner share of mayoral vote'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'nyc_three_mayors_by_swing.png', dpi=150); print('saved figure')

saved figure


## Limitations

- **GDELT tone is a crude lexicon score** on article text; it does not distinguish an article attacking a candidate from one reporting attacks on them, and it is not a measure of what voters saw. Domain restriction captures each outlet's website, not its broadcast. 2021 is the only comparable cycle.
- **IE totals are as reported to the CFB** through the current filing; spending reported under "not determined" or on ballot proposals is excluded. Federal-style dark money outside CFB jurisdiction is not captured.
- **2017 and 2021 results are crosswalked** from the 2012 district lines; the ≥ 50-vote filter in every contest drops ~10% of districts, disproportionately low-turnout ones.
- **2013 is not on disk** (PDF only).